# ⚡ Week 3 — Dimensionality Reduction (PCA) on Steel Industry Energy Data
### Building on the Week 2 Tuned XGBoost Regressor (`Usage_kWh`)

**Goal:** Run a full PCA pipeline on the same feature set used in Week 2, evaluate how many
components are needed to retain 95% of the variance, compare model performance at 3
components vs 95%-variance components vs the original full-feature model, visualize PCA
loadings, and save a single deployable pipeline (`model.joblib`) that takes **raw** input
and outputs a `Usage_kWh` prediction — ready for the Week 3 FastAPI app.

**Note on columns:** This notebook rebuilds the Week 2 feature engineering from the raw
dataset (rather than depending on the Week 2 notebook's runtime state), so it can run
standalone. Raw columns assumed (UCI Steel Industry Energy dataset): `date`, `Usage_kWh`,
`Lagging_Current_Reactive.Power_kVarh`, `Leading_Current_Reactive_Power_kVarh`,
`CO2(tCO2)`, `Lagging_Current_Power_Factor`, `Leading_Current_Power_Factor`, `NSM`,
`WeekStatus`, `Day_of_week`, `Load_Type`. Adjust `RAW_COLS` below if your file differs.

---
## 1️⃣ Imports
---

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', None)

DATA_PATH = "/content/Week 2 (DataSet).xlsx"
RANDOM_STATE = 42


---
## 2️⃣ Load Raw Data + Feature Engineering Function
**- Highlight:** `engineer_features()` reproduces the exact Week 2 feature engineering
(cyclical hour/month encoding, `is_weekend`, dropping `date`/`hour`/`month`/`weekday`) as a
standalone function.

**⚠️ Important fix:** this function is written to its own file, `feature_engineering.py`,
via `%%writefile` (next cell) instead of being defined inline in the notebook. If it were
defined inline, joblib would pickle it as belonging to the notebook's `__main__` module --
and later, when `main.py` (a *different* `__main__`) tries to `joblib.load()` the saved
pipeline, Python won't be able to find it there, causing:
`AttributeError: Can't get attribute 'engineer_features' on <module '__main__' ...>`.
By keeping it in an importable `.py` file (imported the same way in both the notebook and
the FastAPI app), the pickled reference resolves correctly in both places.
---

In [ ]:
%%writefile feature_engineering.py
"""
Shared feature-engineering function used inside the saved model.joblib pipeline.

CRITICAL: This exact file (feature_engineering.py) must also be copied into the FastAPI
app folder, next to main.py, before loading model.joblib there. joblib/pickle stores a
*reference* to this function (module name + function name), not its code -- if the
module isn't importable at load time with the same name, unpickling fails with:
    AttributeError: Can't get attribute 'engineer_features' on <module '__main__' ...>
That's why this is NOT defined inline in the notebook (which would bind it to
'__main__', a module main.py doesn't share) -- it's written to its own .py file instead.
"""

import numpy as np
import pandas as pd


def engineer_features(df):
    """Raw dataframe (with 'date' + raw columns) -> engineered feature dataframe.
    Deterministic, no fitting required -- safe to apply identically to train/test/raw
    single-row inference input.
    """
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])

    hour = df['date'].dt.hour
    month = df['date'].dt.month
    weekday = df['date'].dt.dayofweek

    df['day'] = df['date'].dt.day
    df['is_weekend'] = (weekday >= 5).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['month_sin'] = np.sin(2 * np.pi * month / 12)
    df['month_cos'] = np.cos(2 * np.pi * month / 12)

    df = df.drop(columns=['date'])
    return df


In [ ]:
CATEGORICAL_COLS = ['Load_Type', 'WeekStatus', 'Day_of_week']
NUMERIC_RAW_COLS = [
    'Lagging_Current_Reactive.Power_kVarh',
    'Leading_Current_Reactive_Power_kVarh',
    'CO2(tCO2)',
    'Lagging_Current_Power_Factor',
    'Leading_Current_Power_Factor',
    'NSM',
]

# Import from the .py file just written above (NOT a locally-defined function) --
# this makes the pickled pipeline reference "feature_engineering.engineer_features",
# which is what the FastAPI app will also import from its own copy of the same file.
from feature_engineering import engineer_features

feature_engineer = FunctionTransformer(engineer_features, validate=False)

ENGINEERED_NUMERIC_COLS = NUMERIC_RAW_COLS + ['day', 'is_weekend',
                                               'hour_sin', 'hour_cos',
                                               'month_sin', 'month_cos']


In [ ]:
df_raw = pd.read_excel(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()


---
## 3️⃣ Train-Test Split (on raw data, before any fitting)
**- Highlight:** Splitting happens on the **raw** dataframe first, so every fitted step
downstream (encoder, scaler, PCA, model) only ever sees the training fold — the same
leak-free discipline as Week 2.
---

In [ ]:
X_raw = df_raw.drop(columns=['Usage_kWh'])
y = df_raw['Usage_kWh']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train_raw.shape} | Test: {X_test_raw.shape}")


---
## 4️⃣ Preprocessing Pipeline (Feature Engineering → One-Hot → Scale)
**- Highlight:** `ColumnTransformer` one-hot encodes the 3 categoricals and standard-scales
the numeric/engineered columns in a single fitted object — fit on train only, then used to
transform both train and test. This replaces Week 2's manual `pd.get_dummies` with a
transformer that can be reused safely on a single raw row at inference time.
---

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CATEGORICAL_COLS),
    ('num', StandardScaler(), ENGINEERED_NUMERIC_COLS),
])

# Engineer features first (deterministic), then fit the preprocessor on train only
X_train_eng = engineer_features(X_train_raw)
X_test_eng = engineer_features(X_test_raw)

preprocessor.fit(X_train_eng)

X_train_proc = preprocessor.transform(X_train_eng)
X_test_proc = preprocessor.transform(X_test_eng)

feature_names = preprocessor.get_feature_names_out()
print("Total preprocessed feature count:", X_train_proc.shape[1])


---
## 5️⃣ Fit PCA (train only) — Scree Plot
**- Highlight:** `PCA(n_components=n_features)` is fit on the scaled training data only.
The scree plot (bar chart) shows how much variance each individual component explains —
steep drop-off after the first few components is typical for correlated electrical
features (reactive power, power factor, CO2 all move together).
---

In [ ]:
n_features = X_train_proc.shape[1]
pca_full = PCA(n_components=n_features, random_state=RANDOM_STATE)
pca_full.fit(X_train_proc)

explained_var = pca_full.explained_variance_ratio_

plt.figure(figsize=(10, 5))
plt.bar(range(1, n_features + 1), explained_var, color='teal')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot — Explained Variance per Component')
plt.xticks(range(1, n_features + 1))
plt.show()


---
## 6️⃣ Cumulative Explained Variance Curve (95% threshold)
**- Highlight:** The cumulative curve plus a horizontal 95% line makes it immediately
visible how many components are needed to retain most of the information — this number
feeds directly into the "95% variance PCA model" comparison below.
---

In [ ]:
cumulative_var = np.cumsum(explained_var)
n_components_95 = int(np.argmax(cumulative_var >= 0.95) + 1)

plt.figure(figsize=(10, 5))
plt.plot(range(1, n_features + 1), cumulative_var, marker='o', color='darkorange')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance threshold')
plt.axvline(x=n_components_95, color='gray', linestyle=':',
            label=f'{n_components_95} components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance vs. Number of Components')
plt.legend()
plt.show()

print(f"Components needed to reach 95% variance: {n_components_95}")


---
## 7️⃣ Tune the Base XGBoost Regressor (used across all 3 comparisons)
**- Highlight:** `RandomizedSearchCV` is run **once**, on the full preprocessed
(non-PCA) training features, to find the tuned hyperparameters — the same "Tuned XGBoost
Regressor" referenced in Week 2. These exact hyperparameters are then reused for all three
comparison models below, so the comparison isolates the effect of **dimensionality
reduction**, not different hyperparameters.
---

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=20, scoring='neg_root_mean_squared_error', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
search.fit(X_train_proc, y_train)

best_params = search.best_params_
print("Best params:", best_params)
print(f"Best CV RMSE: {-search.best_score_:.4f}")


---
## 8️⃣ Model Comparison — Original vs 3-Component PCA vs 95%-Variance PCA
**- Highlight:** All three models use the **identical** tuned hyperparameters found above.
Only the input feature space changes: (A) full preprocessed features, (B) first 3 PCA
components, (C) however many components reach 95% variance. This isolates PCA's effect on
accuracy from any hyperparameter differences.
---

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    return {
        'MAE': mean_absolute_error(y_te, preds),
        'RMSE': np.sqrt(mean_squared_error(y_te, preds)),
        'R2': r2_score(y_te, preds),
    }, model

results = {}

# (A) Original Week 2 model — full preprocessed features, no PCA
model_original = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, **best_params)
results['Original (full features)'], model_original = evaluate(
    model_original, X_train_proc, y_train, X_test_proc, y_test
)

# (B) 3-component PCA model
pca_3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_train_pca3 = pca_3.fit_transform(X_train_proc)
X_test_pca3 = pca_3.transform(X_test_proc)

model_pca3 = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, **best_params)
results['3-Component PCA'], model_pca3 = evaluate(
    model_pca3, X_train_pca3, y_train, X_test_pca3, y_test
)

# (C) 95%-variance PCA model
pca_95 = PCA(n_components=n_components_95, random_state=RANDOM_STATE)
X_train_pca95 = pca_95.fit_transform(X_train_proc)
X_test_pca95 = pca_95.transform(X_test_proc)

model_pca95 = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, **best_params)
results[f'{n_components_95}-Component PCA (95% variance)'], model_pca95 = evaluate(
    model_pca95, X_train_pca95, y_train, X_test_pca95, y_test
)

comparison_df = pd.DataFrame(results).T[['MAE', 'RMSE', 'R2']].sort_values('RMSE')
comparison_df.round(4)


---
## 9️⃣ PCA Loadings Heatmap (First 3 Components)
**- Highlight:** Shows exactly which original (preprocessed) features drive PC1, PC2, and
PC3 — e.g. if reactive power and power factor columns dominate PC1, that component is
essentially a "load intensity" axis.
---

In [ ]:
loadings = pd.DataFrame(
    pca_3.components_.T,
    index=feature_names,
    columns=['PC1', 'PC2', 'PC3']
)

plt.figure(figsize=(6, max(6, len(feature_names) * 0.35)))
sns.heatmap(loadings, annot=True, cmap='coolwarm', fmt=".2f", center=0)
plt.title('PCA Loadings — Feature Contribution to PC1/PC2/PC3')
plt.tight_layout()
plt.show()


---
## 🔟 Report — Results & Discussion
*(Placeholder — fill in after running the notebook top-to-bottom with the actual printed
numbers.)*

**Summary template:**
- Components needed for 95% variance: **`{n_components_95}`** out of `{n_features}` total
  preprocessed features.
- Original (full-feature) model RMSE / MAE / R²: *fill in from `comparison_df`*.
- 3-component PCA model RMSE / MAE / R²: *fill in* — expect a meaningfully larger error,
  since 3 components is an aggressive compression of a dataset where electrical features
  are correlated but not perfectly redundant.
- 95%-variance PCA model RMSE / MAE / R²: *fill in* — expect this to be close to the
  original model's performance, since it retains almost all the signal.
- PCA loadings show that **`<name the top 2-3 features per PC from the heatmap>`** load
  most heavily onto PC1, meaning ... *(interpret based on actual output)*.
- Trade-off conclusion: PCA reduces dimensionality and training/inference cost, but for
  this dataset the full-feature model is likely still preferred for deployment accuracy
  unless the extra components (95% version) are used, since 3 components alone lose too
  much signal. *(Confirm this against your own printed numbers before finalizing.)*
---

---
## 1️⃣1️⃣ Save Final Deployable Pipeline (`model.joblib`)
**- Highlight:** This single `Pipeline` accepts a **raw** dataframe (with a `date` column
and the original raw column names — exactly what a web form will submit) and internally
runs: feature engineering → one-hot + scaling → PCA (95%-variance components) → tuned
XGBoost. This is what the Week 3 FastAPI app will load and call `.predict()` on directly,
with no manual preprocessing needed in `main.py`.
---

In [ ]:
final_pipeline = Pipeline(steps=[
    ('feature_engineering', feature_engineer),
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=n_components_95, random_state=RANDOM_STATE)),
    ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, **best_params)),
])

# Fit on ALL raw training data (the preprocessor/PCA/model steps only ever fit on the
# training split, never on X_test_raw)
final_pipeline.fit(X_train_raw, y_train)

# Sanity check on held-out test set
final_preds = final_pipeline.predict(X_test_raw)
print("Final pipeline test RMSE:", np.sqrt(mean_squared_error(y_test, final_preds)))
print("Final pipeline test R2  :", r2_score(y_test, final_preds))

joblib.dump(final_pipeline, 'model.joblib')
print("Saved final_pipeline -> model.joblib")


---
## 1️⃣2️⃣ Freeze Exact Environment Versions (for the FastAPI app)
**- Highlight:** `joblib`/pickle-based model files are version-sensitive — a pipeline
saved with one version of `scikit-learn`/`xgboost`/`lightgbm` can fail (or silently
misbehave) when loaded with a different version elsewhere. Rather than guessing which
package versions to install locally, this cell writes the **exact** versions used here to
`requirements_colab.txt`. Download it along with `model.joblib` and
`feature_engineering.py`, and use it to build the local environment so versions match
exactly, byte-for-byte behavior included.
---

In [ ]:
import subprocess

pinned_packages = ['scikit-learn', 'xgboost', 'lightgbm', 'joblib', 'pandas', 'numpy', 'scipy']

lines = []
for pkg in pinned_packages:
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    version = None
    for line in result.stdout.splitlines():
        if line.startswith('Version:'):
            version = line.split(':', 1)[1].strip()
    if version:
        lines.append(f"{pkg}=={version}")
        print(f"{pkg}=={version}")
    else:
        print(f"[WARN] Could not find version for {pkg}")

with open('requirements_colab.txt', 'w') as f:
    f.write('\n'.join(lines) + '\n')

print("\nSaved requirements_colab.txt -- download this alongside model.joblib and "
      "feature_engineering.py, then locally run:\n"
      "    pip install -r requirements_colab.txt\n"
      "in a clean virtual environment (or after uninstalling conflicting versions) "
      "before starting the FastAPI app.")


**Next (Week 3, Part 2):** `model.joblib` is now a self-contained, raw-input-to-prediction
pipeline — the FastAPI app just needs to load it and call `.predict()` on a one-row
DataFrame built from form inputs (including a `date` string it can parse, or the raw
`hour`/`month`/weekday equivalents assembled into a `date`).

**⚠️ Deployment checklist:** copy **both** files into the FastAPI app folder, next to
`main.py`:
- `model.joblib`
- `feature_engineering.py` (written by the `%%writefile` cell above)

Both are required — `model.joblib` alone will fail to load with an `AttributeError` if
`feature_engineering.py` isn't sitting next to `main.py`, since that's where the pickled
pipeline looks up `engineer_features`.